# Contrastive Skill — Qwen3.5-9B (train split)

같은 task에서 나온 (실패, 성공) trajectory 쌍을 대조해 규칙(rule)을 만들고, 그 규칙이 타당한지 확인하는 전체 파이프라인입니다.
target(문제를 푸는 agent)과 analyst(규칙을 만드는 optimizer)는 같은 vLLM 서버의 `Qwen/Qwen3.5-9B`를 씁니다.

| 단계 | 내용 | 결과 (Drive `DRIVE_ROOT/`) |
|---|---|---|
| 0 | SkillOpt 데이터 준비 (`skillopt_harness/`, SkillOpt `fa4ca18`에서 옮겨 온 평가 코드) | — |
| 1 | train split을 **no-skill**, **no-skill + PI(정답을 user turn에 줌)** 로 각각 풂 | `rollouts/nopi/<bench>/`, `rollouts/pi/<bench>/` |
| 2 | 두 결과가 갈린 문제를 쌍으로 묶음 (hindsight: no-PI 실패·PI 성공 / natural: no-PI 성공·PI 실패) | `pairs/pairs/<bench>.jsonl` |
| 3 | 쌍마다 contrastive analyst(`prompts/contrastive_analyst.md`)로 원인과 rule 생성 | `<TAG>/<bench>.jsonl`, `report.html` |
| 4 | rule validity check: 실패했던 쪽과 같은 조건에서 **system prompt에 rule만 추가**해 다시 풀어 맞는지 확인 | `validity_<VTAG>/` |
| 5 | validity를 통과한 rule 중 최대 `N_SKILL_RULES`개를 임의로 골라 skill을 만들고 SkillOpt **test split**에서 채점 (no-skill test와 비교) | `test_<STAG>/`, `test_noskill/` |

- 모든 결과는 Drive에 바로 저장되고, 각 단계는 **이어서 실행**됩니다 (끝난 문항/쌍/job은 다시 돌리지 않음). 세션이 끊기면 1~5번 셀을 다시 실행한 뒤 멈춘 단계부터 이어 가세요.
- 이미 끝난 단계는 건너뛰어도 됩니다. 뒤 단계는 Drive에 있는 앞 단계 결과를 읽습니다.
- 처음에는 `ROLLOUT_LIMIT`, `ANALYST_LIMIT`, `VALIDATE_LIMIT`을 작게 두고 끝까지 한 번 돌려 본 뒤 0으로 바꾸세요.

순서: 1 설정 → 2 Drive → 3 레포 → 4 설치 → 5 서버 → 6 데이터 → 7~9 rollout → 10 쌍 → 11~15 analyst → 16~18 validity → 19~22 test 평가

In [ ]:
# 1. 설정
REPO_URL = "https://github.com/<YOUR_ID>/<YOUR_REPO>.git"   # ← 본인 레포로 바꾸세요
BRANCH = "main"
REPO_DIR = "/content/contrastive-skill"
MODEL_ID = "Qwen/Qwen3.5-9B"
PORT = 8000
BENCHMARKS = ["searchqa", "docvqa", "officeqa", "livemath", "spreadsheetbench"]

EXP = "qwen3.5-9b_train"     # 실험 폴더 이름 (Drive). 설정을 바꿔 새로 돌릴 때만 바꾸세요
SPLIT = "train"              # SkillOpt split (train 400/107/50/35/80 문항: searchqa/docvqa/officeqa/livemath/spreadsheetbench)
ROLLOUT_LIMIT = 0            # 1단계: 벤치마크당 문항 수 (0 = split 전체)
TAG = "analyst_v1"           # 3단계 출력 폴더 (프롬프트를 바꾸면 TAG도 바꾸세요)
ANALYST_LIMIT = 0            # 3단계: 벤치마크당 최대 쌍 수 (0 = 전체; natural 쌍 우선)
VTAG = "v1"                  # 4단계 출력 폴더
VALIDATE_SOURCES = ["hindsight", "natural"]
REPEATS = 3                  # 4단계: rule / 대조군(빈 skill) 각각 반복 횟수
VALIDATE_LIMIT = 0           # 4단계: 벤치마크당 최대 rule 수 (0 = 전체)

DRIVE_ROOT = f"/content/drive/MyDrive/contrastive_skill/{EXP}"
ROLLOUT_DIR = f"{DRIVE_ROOT}/rollouts"
PAIRS_DIR = f"{DRIVE_ROOT}/pairs"
ANALYST_OUT = f"{DRIVE_ROOT}/{TAG}"
VALID_OUT = f"{DRIVE_ROOT}/validity_{VTAG}"
SKILLOPT_DIR = f"{REPO_DIR}/skillopt_harness"

In [ ]:
# 2. Google Drive (결과 보관용)
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("results ->", DRIVE_ROOT)

In [ ]:
# 3. 레포 받기 (이미 있으면 최신으로 갱신). private 레포면 Colab Secrets에 GITHUB_TOKEN을 등록하세요.
import os, subprocess
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 4. 설치 — 새 세션에서 처음 실행할 때는 재시작 없이 바로 5번으로 가면 됩니다.
# (이미 torch를 import한 세션에서 다시 설치했다면 [런타임 > 세션 다시 시작] 후 1~3번부터 다시 실행하세요.)
# torchaudio는 vllm이 끌어오는 torch와 CUDA 빌드가 달라 import 체인을 깨뜨리므로 제거합니다 (SKILLOPT-COLAB-SETUP 11.5).
!pip install -q -U vllm openai json_repair
# validity check용: 레포의 skillopt_harness/ (SkillOpt fa4ca18에서 필요한 부분만 옮겨 온 코드)의 의존성
!pip install -q pyyaml numpy openpyxl azure-identity httpx datasets huggingface_hub pandas pillow
!pip uninstall -y -q torchaudio
!python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# 5. vLLM 서버 기동
import os, subprocess, time, requests, torch
LOG_PATH = "/content/vllm_server.log"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
MAX_MODEL_LEN = 65536 if gpu_gb >= 35 else 32768   # OfficeQA/SpreadsheetBench 일부 쌍은 4~5만 토큰
print(torch.cuda.get_device_name(0), f"{gpu_gb:.0f}GB", "MAX_MODEL_LEN =", MAX_MODEL_LEN)

env = dict(os.environ, VLLM_USE_FLASHINFER_SAMPLER="0")   # Blackwell 샘플러 함정 회피 (11.7)
cmd = ["python", "-m", "vllm.entrypoints.openai.api_server",
       "--model", MODEL_ID, "--served-model-name", MODEL_ID, "--port", str(PORT),
       "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.90",
       "--limit-mm-per-prompt", '{"image": 1}', "--trust-remote-code",
       "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder"]   # OfficeQA agent의 tool 호출용 (validity check)
proc = subprocess.Popen(cmd, stdout=open(LOG_PATH, "w"), stderr=subprocess.STDOUT, env=env, start_new_session=True)

for _ in range(180):
    if proc.poll() is not None:
        break
    try:
        if requests.get(f"http://localhost:{PORT}/v1/models", timeout=2).ok:
            print("server ready"); break
    except Exception:
        pass
    time.sleep(10)
else:
    print("timeout")
if proc.poll() is not None or not requests.get(f"http://localhost:{PORT}/v1/models", timeout=2).ok:
    txt = open(LOG_PATH).read().splitlines()
    eng = [l for l in txt if "EngineCore" in l] or txt
    for l in eng[-80:]:
        print(l)

## 6. 데이터 준비 (SkillOpt split)

`skillopt_harness/data/*_id_split/`의 문항 목록대로 HuggingFace에서 데이터를 받아 `skillopt_harness/data/`에 만듭니다 (이미 있으면 건너뜀).
OfficeQA(`databricks/officeqa`)는 gated라 Colab Secrets에 `HF_TOKEN`이 있어야 하고, HF에서 접근 조건 동의도 필요합니다.

In [ ]:
# 6. 데이터 준비 + target(qwen_chat) 백엔드 연결
import os, subprocess
os.environ["TARGET_QWEN_CHAT_BASE_URL"] = f"http://localhost:{PORT}/v1"
os.environ["TARGET_QWEN_CHAT_API_KEY"] = "EMPTY"
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN") or "")
except Exception:
    pass

SPLIT_DIRS = {  # benchmark -> (materialize script, output split dir)
    "searchqa": ("materialize_searchqa.py", "data/searchqa_split"),
    "spreadsheetbench": ("materialize_spreadsheetbench.py", "data/spreadsheetbench_split"),
    "docvqa": ("materialize_docvqa.py", "data/docvqa/splits"),
    "officeqa": ("materialize_officeqa.py", "data/officeqa_split"),
    "livemath": ("materialize_livemathematicianbench.py", "data/livemathematicianbench_split"),
}
for b in BENCHMARKS:
    script, split_dir = SPLIT_DIRS[b]
    if os.path.isdir(f"{SKILLOPT_DIR}/{split_dir}"):
        print(f"[{b}] data ready")
        continue
    print(f"[{b}] materializing ...")
    subprocess.run(["python", f"scripts/{script}"], cwd=SKILLOPT_DIR, check=True,
                   env=dict(os.environ, PYTHONPATH=SKILLOPT_DIR))

## 7~9. 1단계: no-skill / no-skill + PI rollout

빈 skill로 `SPLIT`의 문항을 풉니다. 설정은 이전 no-skill baseline과 같습니다 (thinking off, temperature 0.7, max_tokens 8000, SkillOpt 기본 config).
PI run은 `skillopt_harness/scripts/eval_only_pi.py`가 정답을 user turn 맨 앞에 붙입니다 (템플릿: `skillopt_harness/pi_config.json`).
중단돼도 같은 셀을 다시 실행하면 끝난 문항은 건너뜁니다. 단, SpreadsheetBench는 `results.jsonl`이 이미 있으면 그대로 반환하므로, 중간에 끊겼다면 해당 벤치마크 폴더를 지우고 다시 돌리세요.

In [ ]:
# 7. no-skill rollout
!python scripts/run_rollouts.py --mode nopi --split {SPLIT} --limit {ROLLOUT_LIMIT} --model {MODEL_ID} --out-root "{ROLLOUT_DIR}/nopi" --benchmarks {" ".join(BENCHMARKS)}

In [ ]:
# 8. no-skill + PI rollout
!python scripts/run_rollouts.py --mode pi --split {SPLIT} --limit {ROLLOUT_LIMIT} --model {MODEL_ID} --out-root "{ROLLOUT_DIR}/pi" --benchmarks {" ".join(BENCHMARKS)}

In [ ]:
# 9. rollout 정확도 (hard)
import json, os
import pandas as pd
rows = []
for b in BENCHMARKS:
    row = {"benchmark": b}
    for mode in ("nopi", "pi"):
        f = f"{ROLLOUT_DIR}/{mode}/{b}/eval_summary.json"
        if os.path.exists(f):
            s = json.load(open(f))
            row[f"{mode}_hard"], row[f"{mode}_n"] = round(100 * s["hard"], 1), s["n_items"]
    rows.append(row)
pd.DataFrame(rows)

## 10. 2단계: 성공/실패 쌍 만들기

두 rollout의 결과가 갈린 문항을 쌍으로 묶고, analyst 입력(`user_message`)을 미리 만들어 `PAIRS_DIR/pairs/<bench>.jsonl`에 저장합니다.
입력에는 agent의 system prompt, 문제 입력, 실패·성공 trajectory가 들어갑니다 (QA 채점 메시지의 정답, SpreadsheetBench 기대값, PI 정답 헤더는 제거).

In [ ]:
# 10. 쌍 생성 (빠름 — 다시 실행하면 새로 만듭니다)
!python scripts/build_pairs.py --nopi-root "{ROLLOUT_DIR}/nopi" --pi-root "{ROLLOUT_DIR}/pi" --out-dir "{PAIRS_DIR}" --benchmarks {" ".join(BENCHMARKS)}

## 11~15. 3단계: contrastive analyst로 rule 생성

쌍마다 `prompts/contrastive_analyst.md`를 system prompt로 Qwen3.5-9B에 넣어 원인 분석과 rule을 JSON으로 받습니다 (thinking off, JSON schema 제약 디코딩).
결과는 `ANALYST_OUT/`에 바로 저장되고, 다시 실행하면 끝난 쌍은 건너뜁니다.

In [ ]:
# 11. 스모크 테스트: 벤치마크당 2쌍, 별도 폴더
!python scripts/run_analyst.py --base-url http://localhost:{PORT}/v1 --model {MODEL_ID} --pairs-dir "{PAIRS_DIR}/pairs" --out-dir "{DRIVE_ROOT}/analyst_smoke" --limit 2 --benchmarks {" ".join(BENCHMARKS)}

In [ ]:
# 12. 본 실행
!python scripts/run_analyst.py --base-url http://localhost:{PORT}/v1 --model {MODEL_ID} --pairs-dir "{PAIRS_DIR}/pairs" --out-dir "{ANALYST_OUT}" --tag {TAG} --limit {ANALYST_LIMIT} --benchmarks {" ".join(BENCHMARKS)}

In [ ]:
# 13. analyst 결과 요약 (+ vLLM 로그 보관)
import shutil
from IPython.display import Markdown, display
display(Markdown(open(f"{ANALYST_OUT}/summary.md", encoding="utf-8").read()))
shutil.copy(LOG_PATH, DRIVE_ROOT)

In [ ]:
# 14. 개별 결과 보기
import json
BENCH, N = "officeqa", 3
rows = [json.loads(l) for l in open(f"{ANALYST_OUT}/{BENCH}.jsonl", encoding="utf-8")]
for r in rows[:N]:
    print("=" * 100)
    print(r["pair_id"], r["source"], "json_ok:", r["json_ok"], r.get("error", ""))
    print(json.dumps(r["parsed"], indent=2, ensure_ascii=False) if r["parsed"] else r["raw"])

In [ ]:
# 15. HTML 뷰어 생성 (ANALYST_OUT/report.html — Drive에서 내려받아 브라우저로 열기)
!python scripts/render_html.py "{ANALYST_OUT}" --pairs "{PAIRS_DIR}/pairs"

## 16~18. 4단계: rule validity check

**최소 조건**: rule이 타당하다면, 실패했던 쪽의 system prompt에 그 rule만 추가했을 때 같은 문제를 맞혀야 합니다.

- 실패했던 trajectory와 **같은 조건**으로 다시 풉니다. hindsight 쌍은 no-PI, natural 쌍은 PI(정답을 user turn에 준 상태)입니다.
- SkillOpt는 skill 문서를 system prompt의 `## Skill` 섹션에 넣으므로, skill을 rule 한 줄로만 채우면 "원래 system prompt + rule"이 됩니다.
- rule마다 `REPEATS`번 풀고, 대조군으로 같은 조건·빈 skill로 `REPEATS`번 풉니다 (temperature 0.7이라 rule 없이도 맞을 수 있음).
- 판정: `valid = rule 통과 수 > 대조군 통과 수` (그리고 1번 이상 통과).
- 결과는 `VALID_OUT/`에 저장되고, 끝난 job은 다시 돌리지 않습니다.

In [ ]:
# 16. validity check 실행
!python scripts/validate_rules.py --analyst-dir "{ANALYST_OUT}" --pairs-dir "{PAIRS_DIR}/pairs" --out "{VALID_OUT}" --split {SPLIT} --model {MODEL_ID} --benchmarks {" ".join(BENCHMARKS)} --sources {" ".join(VALIDATE_SOURCES)} --repeats {REPEATS} --limit {VALIDATE_LIMIT}

In [ ]:
# 17. 요약
from IPython.display import Markdown, display
display(Markdown(open(f"{VALID_OUT}/summary.md", encoding="utf-8").read()))

In [ ]:
# 18. rule별 결과
import json
import pandas as pd
rows = [json.loads(l) for l in open(f"{VALID_OUT}/rule_validity.jsonl", encoding="utf-8")]
df = pd.DataFrame(rows)[["benchmark", "source", "pair_id", "rule_pass", "control_pass", "valid", "complete", "rule"]]
pd.set_option("display.max_colwidth", 200)
df.sort_values(["benchmark", "valid"], ascending=[True, False])

## 19~22. 5단계: validated rule로 만든 skill을 test split에서 채점

- 벤치마크마다 4단계에서 `valid`가 된 rule 중 최대 `N_SKILL_RULES`개를 `SKILL_SEED`로 임의로 골라 skill 문서(`- rule` 한 줄씩)를 만듭니다. rule은 벤치마크별로 나왔으므로 skill도 벤치마크별입니다. valid rule이 없는 벤치마크는 건너뜁니다.
- skill은 SkillOpt가 system prompt의 `## Skill` 섹션에 넣습니다. 그 외 설정은 1단계와 같습니다 (no-PI, thinking off, temperature 0.7, max_tokens 8000).
- 비교 기준으로 같은 test split을 빈 skill로도 풉니다 (`test_noskill/`, 한 번 돌리면 다른 `STAG`에서도 재사용).
- 고른 rule은 `test_<STAG>/skills/`에 저장되고, 이미 있으면 다시 뽑지 않습니다 (다시 뽑으려면 `STAG`를 바꾸세요). 채점도 이어서 실행됩니다.
- test 문항 수: searchqa 1400 / docvqa 374 / officeqa 172 / livemath 125 / spreadsheetbench 280. 처음엔 `TEST_LIMIT`을 작게 두세요.

In [ ]:
# 19. validated rule 선택 → 벤치마크별 skill 파일
import json, os, random
N_SKILL_RULES = 10      # 벤치마크당 최대 rule 수
SKILL_SEED = 0
STAG = f"n{N_SKILL_RULES}_s{SKILL_SEED}"
TEST_LIMIT = 0          # 벤치마크당 test 문항 수 (0 = 전체)
RUN_NOSKILL_TEST = True # 비교용 no-skill test도 돌릴지
TEST_OUT = f"{DRIVE_ROOT}/test_{STAG}"
SKILL_DIR = f"{TEST_OUT}/skills"
os.makedirs(SKILL_DIR, exist_ok=True)

valid = [json.loads(l) for l in open(f"{VALID_OUT}/rule_validity.jsonl", encoding="utf-8")]
valid = [r for r in valid if r["valid"]]
for b in BENCHMARKS:
    path = f"{SKILL_DIR}/{b}.md"
    if os.path.exists(path):
        print(f"[{b}] keeping existing skill ({path})")
        continue
    pool = sorted((r for r in valid if r["benchmark"] == b), key=lambda r: r["pair_id"])
    if not pool:
        print(f"[{b}] no valid rules -> skipped")
        continue
    picked = random.Random(SKILL_SEED).sample(pool, min(N_SKILL_RULES, len(pool)))
    with open(path, "w", encoding="utf-8") as f:
        f.write("".join(f"- {r['rule']}\n" for r in picked))
    with open(f"{SKILL_DIR}/{b}_selected.json", "w", encoding="utf-8") as f:
        json.dump(picked, f, ensure_ascii=False, indent=2)
    print(f"[{b}] {len(picked)}/{len(pool)} valid rules -> {path}")
for b in BENCHMARKS:
    if os.path.exists(f"{SKILL_DIR}/{b}.md"):
        print()
        print(f"----- {b} -----")
        print(open(f"{SKILL_DIR}/{b}.md", encoding="utf-8").read())

In [ ]:
# 20. test split 채점: rule skill (+ 비교용 no-skill)
!python scripts/run_rollouts.py --mode nopi --split test --limit {TEST_LIMIT} --model {MODEL_ID} --skill-dir "{SKILL_DIR}" --out-root "{TEST_OUT}" --benchmarks {" ".join(BENCHMARKS)}
if RUN_NOSKILL_TEST:
    !python scripts/run_rollouts.py --mode nopi --split test --limit {TEST_LIMIT} --model {MODEL_ID} --out-root "{DRIVE_ROOT}/test_noskill" --benchmarks {" ".join(BENCHMARKS)}

In [ ]:
# 21. 결과 비교 (hard, %)
import json, os
import pandas as pd
rows = []
for b in BENCHMARKS:
    row = {"benchmark": b}
    sel = f"{SKILL_DIR}/{b}_selected.json"
    row["n_rules"] = len(json.load(open(sel, encoding="utf-8"))) if os.path.exists(sel) else 0
    for name, root in (("noskill", f"{DRIVE_ROOT}/test_noskill"), ("skill", TEST_OUT)):
        f = f"{root}/{b}/eval_summary.json"
        if os.path.exists(f):
            s = json.load(open(f))
            row[f"{name}_hard"], row[f"{name}_n"] = round(100 * s["hard"], 1), s["n_items"]
    if "noskill_hard" in row and "skill_hard" in row:
        row["delta"] = round(row["skill_hard"] - row["noskill_hard"], 1)
    rows.append(row)
df = pd.DataFrame(rows)
df.to_csv(f"{TEST_OUT}/comparison.csv", index=False)
df

In [ ]:
# 22. 같은 문항끼리 비교: no-skill → skill 에서 틀림→맞음 / 맞음→틀림 문항 수
import json, os
def load(root, b):
    f = f"{root}/{b}/results.jsonl"
    if not os.path.exists(f):
        return {}
    return {str(r["id"]): r.get("hard") in (1, True, 1.0) for r in map(json.loads, open(f, encoding="utf-8")) }
for b in BENCHMARKS:
    a, s = load(f"{DRIVE_ROOT}/test_noskill", b), load(TEST_OUT, b)
    common = set(a) & set(s)
    if not common:
        continue
    fixed = sum(1 for i in common if not a[i] and s[i])
    broke = sum(1 for i in common if a[i] and not s[i])
    print(f"{b:18s} n={len(common):4d}  fixed(0→1)={fixed:4d}  broke(1→0)={broke:4d}  net={fixed - broke:+d}")